# Advanced Problem Set: Iterators as Function Arguments

**Topic:** iterator exhaustion, single-pass vs multi-pass data processing, safe iterator APIs, and practical patterns for files/generators.

This notebook is self-contained. It uses small in-memory data so every problem can be run without external files.

## Best-practice rules used in this notebook

1. Treat `Iterator` objects as **single-use streams**.
2. Treat `Iterable` objects as possibly reusable, but do not assume that unless the API promises it.
3. If an algorithm needs two passes, either:
   - require a re-iterable input,
   - materialize the stream intentionally,
   - use a one-pass algorithm, or
   - use `itertools.tee` with a clear memory warning.
4. Make consumption visible with tests.
5. Prefer small helper functions with type hints and explicit contracts.


In [1]:
from __future__ import annotations

import csv
import itertools as it
import random
from dataclasses import dataclass
from io import StringIO
from statistics import mean
from typing import Any, Callable, Iterable, Iterator, TypeVar

T = TypeVar("T")
U = TypeVar("U")


## Shared mini dataset

The original lesson used a cars CSV file. For practice, this notebook uses a compact in-memory CSV sample with the same kind of fields: car name, MPG, cylinders, model year, and origin.


In [2]:
CARS_CSV = """Car;MPG;Cylinders;Model;Origin
STRING;DOUBLE;INT;INT;CAT
Chevrolet Chevelle Malibu;18.0;8;70;US
Buick Skylark 320;15.0;8;70;US
Toyota Corolla Mark ii;24.0;4;70;Japan
Datsun PL510;27.0;4;70;Japan
Volkswagen 1131 Deluxe Sedan;26.0;4;70;Europe
Hi 1200D;9.0;8;70;US
Ford Pinto;25.0;4;71;US
Volkswagen Super Beetle 117;0;4;71;Europe
Peugeot 304;30.0;4;71;Europe
Datsun 1200;35.0;4;71;Japan
Chevrolet Impala;13.0;8;72;US
Mazda RX2 Coupe;19.0;3;72;Japan
Renault 12 (sw);26.0;4;72;Europe
Toyota Corolla;31.0;4;74;Japan
Honda Civic CVCC;33.0;4;75;Japan
Volkswagen Rabbit Custom Diesel;43.1;4;78;Europe
Mazda GLC Deluxe;32.8;4;78;Japan
Chevrolet Chevette;30.0;4;78;US
Mazda GLC;46.6;4;80;Japan
Honda Civic 1500 gl;44.6;4;80;Japan
Volkswagen Pickup;44.0;4;82;Europe
Ford Ranger;28.0;4;82;US
"""

@dataclass(frozen=True)
class CarRecord:
    car: str
    mpg: float
    cylinders: int
    model: int
    origin: str


def car_rows(text: str = CARS_CSV) -> Iterator[str]:
    """Yield raw data rows, skipping the header and type row."""
    f = StringIO(text)
    next(f)  # header
    next(f)  # type row
    yield from f


def parse_car_row(row: str) -> CarRecord:
    car, mpg, cylinders, model, origin = row.strip().split(";")
    return CarRecord(car, float(mpg), int(cylinders), int(model), origin)


def parsed_cars(text: str = CARS_CSV) -> Iterator[CarRecord]:
    return (parse_car_row(row) for row in car_rows(text))

# Smoke test
sample = list(parsed_cars())[:3]
sample


[CarRecord(car='Chevrolet Chevelle Malibu', mpg=18.0, cylinders=8, model=70, origin='US'),
 CarRecord(car='Buick Skylark 320', mpg=15.0, cylinders=8, model=70, origin='US'),
 CarRecord(car='Toyota Corolla Mark ii', mpg=24.0, cylinders=4, model=70, origin='Japan')]

## Problem 1 — Diagnose and repair a two-pass bug

You are given a self-iterator that produces random integers. The function below tries to compute `min` and `max` separately, but it fails for iterators because the first pass exhausts the input.

**Tasks**

1. Reproduce the bug.
2. Implement `min_max_single_pass(values)` that returns `(minimum, maximum)` using one pass only.
3. Raise `ValueError("empty iterable")` for empty inputs.


In [3]:
class Randoms:
    """A single-use random-number iterator."""

    def __init__(self, n: int, low: int = 0, high: int = 100) -> None:
        self.n = n
        self.low = low
        self.high = high
        self.i = 0

    def __iter__(self) -> "Randoms":
        return self

    def __next__(self) -> int:
        if self.i >= self.n:
            raise StopIteration
        self.i += 1
        return random.randint(self.low, self.high)


def min_then_max_bad(values: Iterable[int]) -> tuple[int, int]:
    return min(values), max(values)

random.seed(0)
values = Randoms(5)
try:
    print(min_then_max_bad(values))
except ValueError as exc:
    print(type(exc).__name__, exc)


def min_max_single_pass(values: Iterable[T]) -> tuple[T, T]:
    iterator = iter(values)
    try:
        first = next(iterator)
    except StopIteration as exc:
        raise ValueError("empty iterable") from exc

    smallest = largest = first
    for item in iterator:
        if item < smallest:
            smallest = item
        if item > largest:
            largest = item
    return smallest, largest

random.seed(0)
values = Randoms(5)
answer = min_max_single_pass(values)
print(answer)
assert answer == (5, 97)

try:
    min_max_single_pass(iter(()))
except ValueError as exc:
    assert str(exc) == "empty iterable"


ValueError max() iterable argument is empty
(5, 97)


## Problem 2 — Make a reusable iterable instead of a self-iterator

A class whose `__iter__` returns `self` is usually a single-use iterator. Design a class `RandomBatch` whose `__iter__` returns a **fresh generator** each time, so the object can be iterated multiple times.

**Tasks**

1. Implement `RandomBatch`.
2. Make iteration reproducible by storing a seed.
3. Show that `min(batch)` and `max(batch)` both work because each call gets a fresh iterator.


In [4]:
class RandomBatch:
    """A reusable iterable: every iteration starts from the same seed."""

    def __init__(self, n: int, seed: int, low: int = 0, high: int = 100) -> None:
        self.n = n
        self.seed = seed
        self.low = low
        self.high = high

    def __iter__(self) -> Iterator[int]:
        rng = random.Random(self.seed)
        for _ in range(self.n):
            yield rng.randint(self.low, self.high)

batch = RandomBatch(10, seed=0)
print(list(batch))
print(min(batch), max(batch))

assert list(batch) == [49, 97, 53, 5, 33, 65, 62, 51, 100, 38]
assert min(batch) == 5
assert max(batch) == 100


[49, 97, 53, 5, 33, 65, 62, 51, 100, 38]
5 100


## Problem 3 — Detect when an argument is probably single-use

Write a helper `is_iterator(obj)` that returns `True` when `obj` is an iterator, i.e. when `iter(obj) is obj`.

Then write `ensure_reiterable(data)`:

- If `data` is already re-iterable, return it unchanged and `False`.
- If `data` is an iterator, materialize it into a list and return that list and `True`.

This is useful when an API internally needs more than one pass.


In [5]:
def is_iterator(obj: Any) -> bool:
    """Return True for objects that are their own iterator."""
    try:
        return iter(obj) is obj
    except TypeError:
        return False


def ensure_reiterable(data: Iterable[T]) -> tuple[Iterable[T], bool]:
    """
    Return a re-iterable version of data.

    The boolean indicates whether materialization was required.
    """
    if is_iterator(data):
        return list(data), True
    return data, False

numbers_list = [3, 1, 2]
numbers_iter = iter([3, 1, 2])

safe_list, copied_list = ensure_reiterable(numbers_list)
safe_iter, copied_iter = ensure_reiterable(numbers_iter)

print(is_iterator(numbers_list), copied_list, list(safe_list), list(safe_list))
print(is_iterator(numbers_iter), copied_iter, list(safe_iter), list(safe_iter))

assert copied_list is False
assert copied_iter is True
assert min(safe_iter) == 1 and max(safe_iter) == 3


False False [3, 1, 2] [3, 1, 2]
True True [3, 1, 2] [3, 1, 2]


## Problem 4 — Normalize MPG values safely

A common bug is this pattern:

```python
mpg_max = max(record.mpg for record in records)
for record in records:
    ...
```

It works for a list but fails for a single-use iterator.

**Tasks**

1. Implement `mpg_percentages(records)` that returns `(car, percentage_of_max_mpg)`.
2. The function may accept any iterable, including an iterator.
3. Treat `mpg == 0` as missing data and exclude those rows from the maximum calculation and output.
4. Use only one explicit materialization point, and explain it in the docstring.


In [6]:
def mpg_percentages(records: Iterable[CarRecord]) -> list[tuple[str, float]]:
    """
    Return each car's MPG as a percentage of the maximum valid MPG.

    We intentionally materialize only the filtered records because the calculation
    needs the maximum before it can emit percentages. This is safer than
    accidentally making two passes over a possibly single-use iterator.
    """
    valid = [record for record in records if record.mpg > 0]
    if not valid:
        return []

    max_mpg = max(record.mpg for record in valid)
    return [(record.car, record.mpg / max_mpg * 100) for record in valid]

result = mpg_percentages(parsed_cars())
for car, pct in result[:7]:
    print(f"{car:35s} {pct:6.2f}%")

assert result[-3:] == [
    ("Honda Civic 1500 gl", 44.6 / 46.6 * 100),
    ("Volkswagen Pickup", 44.0 / 46.6 * 100),
    ("Ford Ranger", 28.0 / 46.6 * 100),
]
assert all(pct > 0 for _, pct in result)


Chevrolet Chevelle Malibu            38.63%
Buick Skylark 320                    32.19%
Toyota Corolla Mark ii               51.50%
Datsun PL510                         57.94%
Volkswagen 1131 Deluxe Sedan         55.79%
Hi 1200D                             19.31%
Ford Pinto                           53.65%


## Problem 5 — Use `itertools.tee`, but know the tradeoff

`itertools.tee(iterable, n)` can split one stream into independent iterators. It is helpful, but if one copy advances far ahead of another, Python must cache the unseen values.

**Tasks**

1. Implement `min_max_with_tee(values)` using two tee'd iterators.
2. Demonstrate that it works on a single-use iterator.
3. Explain in a comment when you would prefer `min_max_single_pass` instead.


In [7]:
def min_max_with_tee(values: Iterable[T]) -> tuple[T, T]:
    left, right = it.tee(values, 2)
    # This performs two logical passes. If left and right are consumed unevenly,
    # tee may cache many elements. Prefer min_max_single_pass for large streams.
    return min(left), max(right)

random.seed(0)
values = Randoms(10)
answer = min_max_with_tee(values)
print(answer)
assert answer == (5, 100)


(5, 100)


## Problem 6 — Work safely with infinite iterators

Some iterators never end. Calling `min`, `max`, `list`, or `sum` directly on them can run forever.

**Tasks**

1. Create an infinite stream of sensor readings.
2. Implement `take_stats(readings, n)` that consumes only the first `n` values.
3. Return count, min, max, and average.


In [8]:
def sensor_readings(seed: int = 42) -> Iterator[float]:
    rng = random.Random(seed)
    while True:
        baseline = 20.0
        noise = rng.uniform(-2.5, 2.5)
        spike = rng.choice([0, 0, 0, 8])
        yield round(baseline + noise + spike, 2)


def take_stats(readings: Iterable[float], n: int) -> dict[str, float]:
    if n <= 0:
        raise ValueError("n must be positive")

    sample = list(it.islice(readings, n))
    if len(sample) < n:
        raise ValueError("not enough values")

    return {
        "count": len(sample),
        "min": min(sample),
        "max": max(sample),
        "mean": mean(sample),
    }

stats = take_stats(sensor_readings(), 8)
print(stats)
assert stats["count"] == 8


{'count': 8, 'min': 17.65, 'max': 28.31, 'mean': 20.81}


## Problem 7 — Peek without losing data

Sometimes you need to inspect the first item to decide how to process a stream. Calling `next(iterator)` consumes it.

**Tasks**

1. Implement `peek(iterable, default=None)`.
2. Return `(first_item, rebuilt_iterator)`.
3. If the iterable is empty, return `(default, iter(()))`.
4. The rebuilt iterator must include the first item again.


In [9]:
def peek(iterable: Iterable[T], default: U | None = None) -> tuple[T | U | None, Iterator[T]]:
    iterator = iter(iterable)
    try:
        first = next(iterator)
    except StopIteration:
        return default, iter(())
    return first, it.chain([first], iterator)

first, rebuilt = peek(x * x for x in range(5))
print(first)
print(list(rebuilt))

missing, rebuilt_empty = peek(iter(()), default="EMPTY")
print(missing, list(rebuilt_empty))

assert first == 0
assert list(peek([10, 20, 30])[1]) == [10, 20, 30]
assert missing == "EMPTY"


0
[0, 1, 4, 9, 16]
EMPTY []


## Problem 8 — Avoid the `groupby` shared-iterator trap

`itertools.groupby` yields `(key, group_iterator)` pairs. Each `group_iterator` shares the same underlying stream. If you store the group iterators and consume them later, they may be empty or incomplete.

**Tasks**

1. Sort car records by `origin`.
2. Show the bad pattern: saving group iterators for later.
3. Implement the safe version that summarizes each group immediately.


In [10]:
records = sorted(parsed_cars(), key=lambda r: r.origin)

# Bad pattern: save group iterators for later.
saved_groups = [(origin, group) for origin, group in it.groupby(records, key=lambda r: r.origin)]
print("Bad saved group sizes:", [(origin, len(list(group))) for origin, group in saved_groups])

# Good pattern: consume each group immediately inside the groupby loop.
def mpg_by_origin(records: Iterable[CarRecord]) -> dict[str, dict[str, float]]:
    sorted_records = sorted(records, key=lambda r: r.origin)
    summary: dict[str, dict[str, float]] = {}

    for origin, group in it.groupby(sorted_records, key=lambda r: r.origin):
        valid = [record.mpg for record in group if record.mpg > 0]
        summary[origin] = {
            "count": len(valid),
            "min": min(valid),
            "max": max(valid),
            "mean": mean(valid),
        }
    return summary

summary = mpg_by_origin(parsed_cars())
for origin, stats in summary.items():
    print(origin, stats)

assert set(summary) == {"Europe", "Japan", "US"}
assert summary["Japan"]["max"] == 46.6


Bad saved group sizes: [('Europe', 0), ('Japan', 0), ('US', 0)]
Europe {'count': 5, 'min': 26.0, 'max': 44.0, 'mean': 33.82}
Japan {'count': 9, 'min': 19.0, 'max': 46.6, 'mean': 32.55555555555556}
US {'count': 7, 'min': 9.0, 'max': 30.0, 'mean': 19.714285714285715}


## Problem 9 — Build a lazy pipeline and summarize it once

`map`, `filter`, and generator expressions produce iterators. Reusing the same pipeline object for multiple aggregations is a common mistake.

**Tasks**

1. Build a lazy pipeline of valid MPG values for Japanese cars.
2. Demonstrate why separate `min()` and `max()` calls on the same pipeline fail.
3. Implement `summarize_once(values)` using one pass.


In [11]:
def summarize_once(values: Iterable[float]) -> dict[str, float]:
    iterator = iter(values)
    try:
        first = next(iterator)
    except StopIteration as exc:
        raise ValueError("empty iterable") from exc

    count = 1
    total = first
    smallest = largest = first

    for value in iterator:
        count += 1
        total += value
        if value < smallest:
            smallest = value
        if value > largest:
            largest = value

    return {
        "count": count,
        "min": smallest,
        "max": largest,
        "mean": total / count,
    }

pipeline = (
    record.mpg
    for record in parsed_cars()
    if record.origin == "Japan" and record.mpg > 0
)

try:
    print("bad min/max:", min(pipeline), max(pipeline))
except ValueError as exc:
    print("bad pipeline failed:", exc)

pipeline = (
    record.mpg
    for record in parsed_cars()
    if record.origin == "Japan" and record.mpg > 0
)
answer = summarize_once(pipeline)
print(answer)

assert answer["count"] == 9
assert answer["max"] == 46.6


bad pipeline failed: max() iterable argument is empty
{'count': 9, 'min': 19.0, 'max': 46.6, 'mean': 32.55555555555556}


## Problem 10 — Design a re-iterable file-like data source

A file object is an iterator. After reading it once, it is exhausted unless you seek back to the beginning or reopen it.

Design a reusable class `CarTable` whose `__iter__` creates a new `StringIO` and `csv.DictReader` every time. This models a best-practice API: the table object is re-iterable, but each individual iterator is still single-use.

**Tasks**

1. Implement `CarTable`.
2. Make it skip the type row.
3. Show that multiple aggregations work on the same `CarTable` object.


In [12]:
class CarTable:
    def __init__(self, text: str) -> None:
        self.text = text

    def __iter__(self) -> Iterator[CarRecord]:
        f = StringIO(self.text)
        reader = csv.DictReader(f, delimiter=";")
        next(reader)  # skip type row
        for row in reader:
            yield CarRecord(
                car=row["Car"],
                mpg=float(row["MPG"]),
                cylinders=int(row["Cylinders"]),
                model=int(row["Model"]),
                origin=row["Origin"],
            )

table = CarTable(CARS_CSV)

max_mpg = max(record.mpg for record in table if record.mpg > 0)
avg_us_mpg = mean(record.mpg for record in table if record.origin == "US" and record.mpg > 0)
count_europe = sum(1 for record in table if record.origin == "Europe")

print(max_mpg)
print(round(avg_us_mpg, 2))
print(count_europe)

assert max_mpg == 46.6
assert round(avg_us_mpg, 2) == 19.71
assert count_europe == 6


46.6
19.71
6


## Problem 11 — Write a function with an explicit iterator contract

Implement `top_n_by(records, key, n)`.

**Requirements**

1. Accept any iterable, including a generator.
2. Consume the input only once.
3. Return the top `n` items sorted from largest key to smallest key.
4. Do not mutate the original data.

For simplicity, use `sorted` internally; this intentionally materializes because exact top-N sorting needs stored candidates. For very large streams, `heapq.nlargest` would be a more memory-efficient alternative.


In [13]:
def top_n_by(records: Iterable[T], key: Callable[[T], Any], n: int) -> list[T]:
    if n < 0:
        raise ValueError("n must be non-negative")
    if n == 0:
        return []
    return sorted(records, key=key, reverse=True)[:n]

top_5 = top_n_by((record for record in parsed_cars() if record.mpg > 0), key=lambda r: r.mpg, n=5)
for record in top_5:
    print(f"{record.car:35s} {record.mpg:4.1f}")

assert [record.mpg for record in top_5] == [46.6, 44.6, 44.0, 43.1, 35.0]
assert top_n_by(parsed_cars(), key=lambda r: r.mpg, n=0) == []


Mazda GLC                           46.6
Honda Civic 1500 gl                 44.6
Volkswagen Pickup                   44.0
Volkswagen Rabbit Custom Diesel     43.1
Datsun 1200                         35.0


## Problem 12 — Cap accidental consumption with a debugging iterator

When debugging iterator bugs, it is useful to wrap an iterable and count how many items were consumed.

**Tasks**

1. Implement `CountingIterator`.
2. Wrap a generator of car records.
3. Prove that `summarize_once` consumes exactly the valid MPG values you feed it.


In [14]:
class CountingIterator(Iterator[T]):
    def __init__(self, iterable: Iterable[T]) -> None:
        self._iterator = iter(iterable)
        self.consumed = 0

    def __iter__(self) -> "CountingIterator[T]":
        return self

    def __next__(self) -> T:
        item = next(self._iterator)
        self.consumed += 1
        return item

mpgs = (
    record.mpg
    for record in parsed_cars()
    if record.origin == "Europe" and record.mpg > 0
)
counted = CountingIterator(mpgs)
summary = summarize_once(counted)

print(summary)
print("items consumed:", counted.consumed)

assert counted.consumed == summary["count"] == 5
assert summary["max"] == 44.0


{'count': 5, 'min': 26.0, 'max': 44.0, 'mean': 33.82}
items consumed: 5


## Problem 13 — Final challenge: safe normalization API

Create `normalize(values)` that works for lists, tuples, generators, file iterators, and custom iterables.

**Requirements**

1. Accept numeric values.
2. Ignore `None` values.
3. Return values divided by the maximum valid value.
4. Return a list because normalization requires knowing the maximum before producing output.
5. Raise `ValueError("no valid values")` if no valid numeric values exist.


In [15]:
def normalize(values: Iterable[float | int | None]) -> list[float]:
    valid = [float(value) for value in values if value is not None]
    if not valid:
        raise ValueError("no valid values")

    max_value = max(valid)
    if max_value == 0:
        raise ValueError("maximum value is zero")

    return [value / max_value for value in valid]

print(normalize([2, None, 4, 8]))
print(normalize(x for x in [10, 5, None, 20]))

assert normalize([2, None, 4, 8]) == [0.25, 0.5, 1.0]
assert normalize(x for x in [10, 5, None, 20]) == [0.5, 0.25, 1.0]

for bad in ([None, None], [], [0, 0]):
    try:
        normalize(bad)
    except ValueError as exc:
        print(bad, "->", exc)


[0.25, 0.5, 1.0]
[0.5, 0.25, 1.0]
[None, None] -> no valid values
[] -> no valid values
[0, 0] -> maximum value is zero


## Takeaways

- Passing an iterator to a function that expects an iterable is legal, but it may be dangerous if the function makes multiple passes.
- A self-iterator is usually single-use; a re-iterable object returns a fresh iterator each time.
- Prefer one-pass algorithms for streams.
- Materialize intentionally when the algorithm truly needs earlier values later.
- Use `itertools.tee` carefully because it can cache unseen values.
- Document whether your function consumes its input once, multiple times, or returns a lazy iterator.
